# OHLCV 1m：CCXT（OKX）回补与成交量标定

本 notebook **仅保留 ccxt 路径**，已删除原 Coinglass HTTP 回补段落。

## 成交量约定
- **补丁 CSV**（给 `Data_Pipeline`）：`volume` 列为 **OKX 原始 BTC 数量**（与 ccxt `fetch_ohlcv` 一致）。Pipeline 内依次：`volume × close`（USDT 名义）× **`ratio`**（标定 JSON）。
- **标定**：UTC **2026-03-04** 全日重叠分钟：`ratio = median(V_bin / (BTC_okx × close_okx))`，对比值 1%–99% 截断后取中位数。
- 结果：`data/backfill/okx_binance_volume_ratio.json`。

## 流程
1. 加载 `aligned_1m`
2. `fetch_with_ccxt`（`okx_volume_mode`: `raw_btc` 或 `usd_scaled`）
3. 3/4 标定并写 JSON
4. 缺口检测 → 拉 OKX **raw BTC** 保存 `okx_price_1m_gap_candidate.csv`
5. 可选：预览 merge（先把 raw×close×ratio 再对齐列名）


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import ccxt

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

ALIGNED_1M = ROOT / "data" / "aligned" / "aligned_1m.csv"
BACKFILL_DIR = ROOT / "data" / "backfill"
BACKFILL_DIR.mkdir(parents=True, exist_ok=True)
RATIO_JSON = BACKFILL_DIR / "okx_binance_volume_ratio.json"

price_cols = [
    "futures_price_history_btcusdt_binance__open",
    "futures_price_history_btcusdt_binance__high",
    "futures_price_history_btcusdt_binance__low",
    "futures_price_history_btcusdt_binance__close",
    "futures_price_history_btcusdt_binance__volume_usd",
]

aligned = pd.read_csv(ALIGNED_1M, usecols=["time_ms", "time_utc"] + price_cols)
aligned["time_utc"] = pd.to_datetime(aligned["time_utc"], utc=True, errors="coerce")
aligned["time_ms"] = pd.to_numeric(aligned["time_ms"], errors="coerce")
for c in price_cols:
    aligned[c] = pd.to_numeric(aligned[c], errors="coerce")
aligned = aligned.dropna(subset=["time_ms", "time_utc"]).sort_values("time_ms")
print("aligned rows:", len(aligned), "|", ALIGNED_1M)


In [8]:
def fetch_with_ccxt(
    exchange_id: str,
    symbol: str,
    start_ms: int,
    end_ms: int,
    *,
    limit: int = 300,
    max_loops: int = 120,
    okx_volume_mode: str = "raw_btc",
    okx_volume_scale: float = 1.0,
):
    """
    For OKX: ccxt volume is BTC base amount.
    okx_volume_mode "raw_btc": leave volume as BTC (for patch CSV → Data_Pipeline).
    "usd_scaled": volume = BTC * close * okx_volume_scale (for analysis / preview).
    """
    ex = getattr(ccxt, exchange_id)({"enableRateLimit": True})
    since = int(start_ms)
    end_limit = int(end_ms) + 60_000
    bag = []
    for _ in range(max_loops):
        if since >= end_limit:
            break
        rows = ex.fetch_ohlcv(symbol, "1m", since=since, limit=limit)
        if not rows:
            break
        bag.extend(rows)
        last_ts = int(rows[-1][0])
        if last_ts < since:
            break
        since = last_ts + 60_000
        if len(rows) < limit:
            break
    out = pd.DataFrame(bag, columns=["time_ms", "open", "high", "low", "close", "volume"]).drop_duplicates("time_ms").sort_values("time_ms")
    out = out[(out["time_ms"] >= start_ms) & (out["time_ms"] <= end_ms)].copy()
    out["time_ms"] = pd.to_numeric(out["time_ms"], errors="coerce").astype("int64")
    out["time_utc"] = pd.to_datetime(out["time_ms"], unit="ms", utc=True)
    if str(exchange_id).lower() == "okx":
        v = pd.to_numeric(out["volume"], errors="coerce")
        cl = pd.to_numeric(out["close"], errors="coerce")
        if okx_volume_mode == "usd_scaled":
            out["volume"] = v * cl * float(okx_volume_scale)
        else:
            out["volume"] = v
    return out


In [ ]:
# --- 标定：UTC 2026-03-04（OKX raw BTC × OKX close vs Binance USD volume）---
CAL_UTC = pd.Timestamp("2026-03-04", tz="UTC")
CAL_START_MS = int(CAL_UTC.timestamp() * 1000)
CAL_END_MS = CAL_START_MS + 24 * 60 * 60 * 1000 - 60_000

print("Calibration (UTC):", pd.to_datetime(CAL_START_MS, unit="ms", utc=True), "->", pd.to_datetime(CAL_END_MS, unit="ms", utc=True))

okx_cal = fetch_with_ccxt("okx", "BTC/USDT:USDT", CAL_START_MS, CAL_END_MS, okx_volume_mode="raw_btc")
side = okx_cal[["time_ms", "volume", "close"]].rename(columns={"volume": "vol_btc_okx", "close": "close_okx"})
bin_cal = aligned.loc[(aligned["time_ms"] >= CAL_START_MS) & (aligned["time_ms"] <= CAL_END_MS)].copy()

m = bin_cal.merge(side, on="time_ms", how="inner")
v_bin = m["futures_price_history_btcusdt_binance__volume_usd"]
v_okx_notional = m["vol_btc_okx"] * m["close_okx"]
mask = (v_bin > 0) & (v_okx_notional > 0) & np.isfinite(v_bin) & np.isfinite(v_okx_notional)
rat = (v_bin[mask] / v_okx_notional[mask]).astype(float)
lo, hi = rat.quantile(0.01), rat.quantile(0.99)
rat_f = rat[(rat >= lo) & (rat <= hi)]
OKX_BINANCE_VOL_RATIO = float(rat_f.median()) if len(rat_f) else float(rat.median())

payload = {
    "ratio": OKX_BINANCE_VOL_RATIO,
    "calibration_utc_date": "2026-03-04",
    "n_minutes_overlap": int(mask.sum()),
    "n_ratio_after_trim": int(len(rat_f)),
    "method": "median(V_bin / (BTC_okx*close_okx)) with 1-99pct trim",
}
RATIO_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Saved:", RATIO_JSON)
print(json.dumps(payload, indent=2))


In [ ]:
# --- 缺口检测（最大连续全空 OHLCV）---
all_null = aligned[price_cols].isna().all(axis=1)
gap_df = aligned.loc[all_null, ["time_ms", "time_utc"]].copy()
if gap_df.empty:
    print("No all-null OHLCV rows.")
    TARGET_START_MS = None
    TARGET_END_MS = None
else:
    gap_df["grp"] = (gap_df["time_ms"].diff().fillna(60_000) != 60_000).cumsum()
    segments = (
        gap_df.groupby("grp")
        .agg(start_ms=("time_ms", "min"), end_ms=("time_ms", "max"), rows=("time_ms", "size"))
        .sort_values("rows", ascending=False)
    )
    display(segments.head(5))
    target = segments.iloc[0]
    TARGET_START_MS = int(target["start_ms"])
    TARGET_END_MS = int(target["end_ms"])
    print("Target gap rows:", int(target["rows"]), "|", TARGET_START_MS, "->", TARGET_END_MS)


In [ ]:
# --- 保存 OKX 缺口：volume 列为 raw BTC（供 Data_Pipeline 再 ×close×ratio）---
if RATIO_JSON.exists():
    OKX_BINANCE_VOL_RATIO = float(json.loads(RATIO_JSON.read_text(encoding="utf-8"))["ratio"])
else:
    OKX_BINANCE_VOL_RATIO = 1.0
    print("Warning: ratio JSON missing; preview scale uses 1.0")

ccxt_df = None
if TARGET_START_MS is None:
    print("Skip gap fetch.")
else:
    try:
        ccxt_df = fetch_with_ccxt(
            "okx",
            "BTC/USDT:USDT",
            TARGET_START_MS,
            TARGET_END_MS,
            okx_volume_mode="raw_btc",
        )
    except Exception as e:
        print("OKX fetch failed:", e)
        ccxt_df = None

if ccxt_df is not None and not ccxt_df.empty:
    expected_idx = pd.Index(range(TARGET_START_MS, TARGET_END_MS + 60_000, 60_000), name="time_ms")
    got_idx = pd.Index(ccxt_df["time_ms"].tolist(), name="time_ms")
    print("Coverage:", len(got_idx), "/", len(expected_idx))
    out_csv = BACKFILL_DIR / "okx_price_1m_gap_candidate.csv"
    ccxt_df.to_csv(out_csv, index=False)
    print("Saved (volume=BTC raw):", out_csv)
    display(ccxt_df.head(3))
else:
    print("No gap CSV.")


In [ ]:
# 可选：预览 merge（将 raw BTC × close × ratio 再写入 Binance 列名）
APPLY_CCXT_PATCH_PREVIEW = True

if RATIO_JSON.exists():
    OKX_BINANCE_VOL_RATIO = float(json.loads(RATIO_JSON.read_text(encoding="utf-8"))["ratio"])
else:
    OKX_BINANCE_VOL_RATIO = 1.0

if APPLY_CCXT_PATCH_PREVIEW and ccxt_df is not None and not ccxt_df.empty:
    patch = aligned.copy()
    prev = ccxt_df.copy()
    prev["vol_usd"] = pd.to_numeric(prev["volume"], errors="coerce") * pd.to_numeric(prev["close"], errors="coerce") * float(OKX_BINANCE_VOL_RATIO)
    map_df = prev.rename(
        columns={
            "open": "futures_price_history_btcusdt_binance__open",
            "high": "futures_price_history_btcusdt_binance__high",
            "low": "futures_price_history_btcusdt_binance__low",
            "close": "futures_price_history_btcusdt_binance__close",
            "vol_usd": "futures_price_history_btcusdt_binance__volume_usd",
        }
    )
    patch = patch.merge(map_df[["time_ms"] + price_cols], on="time_ms", how="left", suffixes=("", "__ccxt"))
    for c in price_cols:
        alt = c + "__ccxt"
        patch[c] = patch[c].combine_first(patch[alt])
        patch = patch.drop(columns=[alt])
    out_patch = BACKFILL_DIR / "aligned_1m_patched_with_ccxt_preview.csv"
    patch.to_csv(out_patch, index=False)
    print("Saved patch preview:", out_patch)
    still_null = patch[price_cols].isna().all(axis=1).sum()
    print("All-null OHLCV rows after patch:", int(still_null))
else:
    print("Patch preview skipped.")
